# NSynth Instrument Classification

Classify musical instruments from audio notes using the [NSynth dataset](https://magenta.withgoogle.com/datasets/nsynth).

**Goal**: Given a short audio note, predict which instrument family produced it.

## Dataset
- 305,979 musical notes from 1,006 instruments
- 16kHz mono audio, 4 seconds each (3s held + 1s decay)
- 11 instrument families: bass, brass, flute, guitar, keyboard, mallet, organ, reed, string, synth_lead, vocal
- MIDI pitches 21-108, velocities 25/50/75/100/127

In [ ]:
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

## Constants

In [ ]:
SAMPLE_RATE = 16000
NOTE_DURATION = 4.0  # seconds

INSTRUMENT_FAMILIES = [
    "bass", "brass", "flute", "guitar", "keyboard",
    "mallet", "organ", "reed", "string", "synth_lead", "vocal",
]

INSTRUMENT_SOURCES = ["acoustic", "electronic", "synthetic"]

NOTE_QUALITIES = [
    "bright", "dark", "distortion", "fast_decay", "long_release",
    "multiphonic", "nonlinear_env", "percussive", "reverb", "tempo_synced",
]

DATA_DIR = Path("../data")
TRAIN_DIR = DATA_DIR / "nsynth-train"
VALID_DIR = DATA_DIR / "nsynth-valid"
TEST_DIR = DATA_DIR / "nsynth-test"

## Data Loading

NSynth provides JSON + WAV files. Each JSON file maps note IDs to metadata, and audio is stored as `.wav` files.

In [ ]:
def load_nsynth_json(split_dir: Path) -> pd.DataFrame:
    """Load NSynth metadata from JSON file into a DataFrame."""
    import json

    json_path = split_dir / "examples.json"
    with open(json_path) as f:
        data = json.load(f)

    rows = []
    for note_str, meta in data.items():
        rows.append({
            "note_str": note_str,
            "instrument_family": meta["instrument_family"],
            "instrument_family_str": meta["instrument_family_str"],
            "instrument_source": meta["instrument_source"],
            "instrument_source_str": meta["instrument_source_str"],
            "pitch": meta["pitch"],
            "velocity": meta["velocity"],
            "qualities": meta["qualities"],
        })

    return pd.DataFrame(rows)


def load_audio(note_str: str, split_dir: Path) -> np.ndarray:
    """Load a single audio file and return waveform as numpy array."""
    wav_path = split_dir / "audio" / f"{note_str}.wav"
    audio, _ = librosa.load(wav_path, sr=SAMPLE_RATE)
    return audio

## Data Exploration

Load the dataset and inspect class distribution.

In [ ]:
# Uncomment when data is downloaded:
# df_train = load_nsynth_json(TRAIN_DIR)
# df_valid = load_nsynth_json(VALID_DIR)
# df_test = load_nsynth_json(TEST_DIR)
# print(f"Train: {len(df_train)}, Valid: {len(df_valid)}, Test: {len(df_test)}")

In [ ]:
# def plot_family_distribution(df: pd.DataFrame, title: str = ""):
#     counts = df["instrument_family_str"].value_counts().sort_index()
#     fig, ax = plt.subplots(figsize=(10, 5))
#     counts.plot.bar(ax=ax)
#     ax.set_ylabel("Count")
#     ax.set_title(f"Instrument Family Distribution {title}")
#     plt.tight_layout()
#     plt.show()
#
# plot_family_distribution(df_train, "(Train)")

## Feature Extraction

Extract audio features for classification. Options:
- **Mel spectrogram**: compact time-frequency representation
- **MFCCs**: standard for audio classification
- **Raw waveform**: for neural network approaches

In [ ]:
def extract_mel_spectrogram(audio: np.ndarray, sr: int = SAMPLE_RATE,
                            n_mels: int = 128, hop_length: int = 512) -> np.ndarray:
    """Extract mel spectrogram from audio waveform."""
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=n_mels, hop_length=hop_length)
    return librosa.power_to_db(mel, ref=np.max)


def extract_mfcc(audio: np.ndarray, sr: int = SAMPLE_RATE,
                 n_mfcc: int = 20, hop_length: int = 512) -> np.ndarray:
    """Extract MFCCs from audio waveform."""
    return librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc, hop_length=hop_length)

In [ ]:
# Example: visualize a mel spectrogram
# audio = load_audio(df_train.iloc[0]["note_str"], TRAIN_DIR)
# mel = extract_mel_spectrogram(audio)
#
# fig, ax = plt.subplots(figsize=(10, 4))
# librosa.display.specshow(mel, sr=SAMPLE_RATE, hop_length=512, x_axis="time", y_axis="mel", ax=ax)
# ax.set_title(f"Mel Spectrogram - {df_train.iloc[0]['instrument_family_str']}")
# plt.tight_layout()
# plt.show()

## TODO

Next steps to implement:

1. **Download data**: `nsynth-{train,valid,test}.jsonwav.tar.gz` from [magenta](http://download.magenta.tensorflow.org/datasets/nsynth/)
2. **Dataset class**: PyTorch `Dataset` that loads audio on-the-fly with caching
3. **Baseline model**: simple classifier (logistic regression / small CNN) on mel spectrograms
4. **Training loop**: train, validate, log metrics
5. **Evaluation**: confusion matrix, per-class accuracy, audio samples of misclassifications